In [1]:
pip uninstall -y transformers peft accelerate

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0


In [2]:
pip install "transformers==4.40.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.


In [3]:
pip install "accelerate==0.30.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 7.0 MB/s eta 0:00:00


In [4]:
pip install "peft==0.10.0"   # AGIPの実験には不要なら省略可

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 3.3 MB/s eta 0:00:00


In [5]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator
)
from transformers.pytorch_utils import Conv1D
from datasets import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# =====================================================
# 1. ヘルパー関数
# =====================================================
def is_prunable_module(module):
    return isinstance(module, (nn.Linear, nn.Conv2d, Conv1D))

def importance_based_pruning(module, prune_rate):
    """勾配重要度に基づくプルーニング"""
    if hasattr(module, "weight") and module.weight.grad is not None:
        weight = module.weight.data
        grad = module.weight.grad.data
        importance = torch.abs(weight * grad)

        num_params = importance.numel()
        k = int(prune_rate * num_params)
        if k == 0: return

        threshold = torch.topk(importance.view(-1), k, largest=False).values.max()
        mask = (importance > threshold).float()
        prune.CustomFromMask.apply(module, "weight", mask)

# =====================================================
# 2. カスタムTrainer: AGIP (TAなし)
# =====================================================
class AGIP_NoTA_Trainer(Trainer):
    def __init__(self, *args, base_prune_rate=0.1, prune_interval=500, **kwargs):
        super().__init__(*args, **kwargs)
        self.base_prune_rate = base_prune_rate
        self.prune_interval = prune_interval

        self.global_step_counter = 0
        self.history = []
        self.loss_buffer = []
        self.grad_buffer = []
        self.loss_window = []
        self.grad_norm_window = []

    # create_optimizer はオーバーライドせず、デフォルトのAdamWを使用

    def training_step(self, model, inputs):
        model.train()
        inputs = self._prepare_inputs(inputs)

        # 1. Forward & Loss
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)

        # 2. Backward
        if self.args.n_gpu > 1:
            loss = loss.mean()
        loss.backward()

        # 勾配ノルム計算 (記録用)
        grad_norm_sq = 0.0
        for p in model.parameters():
            if p.grad is not None:
                grad_norm_sq += p.grad.data.norm(2).item() ** 2
        grad_norm = grad_norm_sq ** 0.5

        E = loss.detach().item()
        self.loss_buffer.append(E)
        self.grad_buffer.append(grad_norm)
        self.loss_window.append(E)
        self.grad_norm_window.append(grad_norm)

        self.global_step_counter += 1

        optimizer = self.optimizer

        # 3. 勾配クリッピング (Max=4.0)
        # ※TA版と条件を合わせるため必須
        if self.args.max_grad_norm is not None and self.args.max_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), self.args.max_grad_norm)

        # 4. パラメータ更新 (AdamW)
        optimizer.step()

        # ★重要: ここで TA項による追加更新を行わない (No TA)

        # 5. 動的プルーニング (AGIP)
        if self.global_step_counter % self.prune_interval == 0:
            self.dynamic_pruning(model)

        # 6. 勾配リセット
        optimizer.zero_grad()

        # ログ出力
        if self.global_step_counter % 100 == 0:
            avg_loss = np.mean(self.loss_buffer)
            avg_grad = np.mean(self.grad_buffer)
            self.loss_buffer = []
            self.grad_buffer = []

            val_metrics = self.evaluate(eval_dataset=self.eval_dataset)
            val_loss = val_metrics['eval_loss']
            self.model.train()

            self.history.append({
                "step": self.global_step_counter,
                "train_loss": avg_loss,
                "val_loss": val_loss,
                "grad_norm": avg_grad
            })
            print(f"[Step {self.global_step_counter}] Train Loss: {avg_loss:.4f}, Val Loss: {val_loss:.4f}, Grad: {avg_grad:.4f}")

        return loss.detach()

    def dynamic_pruning(self, model):
        avg_loss = np.mean(self.loss_window[-50:]) if self.loss_window else 0
        avg_grad = np.mean(self.grad_norm_window[-50:]) if self.grad_norm_window else 0

        if avg_loss > 0:
            prune_rate = self.base_prune_rate * (avg_grad / (avg_loss + 1e-6))
        else:
            prune_rate = self.base_prune_rate

        # ★ 上限 30% (他実験と統一)
        prune_rate = min(max(prune_rate, 0.01), 0.3)

        print(f"\n >>> [Pruning Event (No TA)] Step {self.global_step_counter}")
        print(f"     Stats - Loss: {avg_loss:.4f}, Grad: {avg_grad:.4f} => Rate: {prune_rate:.3f}")

        for _, module in model.named_modules():
            if is_prunable_module(module):
                if hasattr(module, "weight_mask"):
                    prune.remove(module, "weight")
                importance_based_pruning(module, prune_rate)

        # スパース率表示
        total_params = 0
        zero_params = 0
        for _, module in model.named_modules():
            if hasattr(module, "weight"):
                total_params += module.weight.nelement()
                zero_params += torch.sum(module.weight == 0).item()
        sparsity = zero_params / total_params if total_params > 0 else 0
        print(f"     Current Sparsity: {sparsity*100:.2f}% ({zero_params}/{total_params})\n")

# =====================================================
# 3. メイン実行部
# =====================================================
def main():
    print("Loading WikiText-2 dataset...")
    data_urls = {
        "train": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt",
        "validation": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt",
        "test": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/test.txt"
    }
    raw_datasets = load_dataset("text", data_files=data_urls)
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    tokenized_datasets = raw_datasets.map(tokenize_function, batched=True, remove_columns=["text"])
    tokenized_datasets = tokenized_datasets.map(lambda batch: {"labels": batch["input_ids"]}, batched=True)
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    print("Loading GPT-2 Model...")
    model = AutoModelForCausalLM.from_pretrained("gpt2")
    model.resize_token_embeddings(len(tokenizer))

    output_dir = "./results_agip_nota_30_clip4"

    training_args = TrainingArguments(
        output_dir=output_dir,
        evaluation_strategy="no",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        learning_rate=5e-5,
        num_train_epochs=1,
        weight_decay=0.01,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
        dataloader_num_workers=2,
        max_grad_norm=4.0  # ★ 4.0 (統一)
    )

    trainer = AGIP_NoTA_Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=default_data_collator,
        base_prune_rate=0.1,
        prune_interval=500
    )

    print("🚀 Starting Training (AGIP only, No TA, 30%, Clip=4.0)...")
    trainer.train()

    print("\n🔍 Evaluating on Test Set...")
    test_results = trainer.evaluate(tokenized_datasets["test"])
    print(f"✅ Final Test Loss: {test_results['eval_loss']:.4f}")

    # ログ保存
    df_log = pd.DataFrame(trainer.history)
    df_log.to_csv("agip_nota_30_clip4_log.csv", index=False)

    print("Finalizing model pruning...")
    for name, module in model.named_modules():
        if is_prunable_module(module) and hasattr(module, "weight_mask"):
            prune.remove(module, "weight")

    model.save_pretrained("./final_gpt2_agip_nota_30_clip4")
    print("✅ Done.")

if __name__ == "__main__":
    main()

Loading WikiText-2 dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

🚀 Starting Training (AGIP only, No TA, 30%, Clip=4.0)...


Step,Training Loss,Validation Loss
99,No log,1.335373
100,1.767300,No Log
199,1.767300,1.275837
200,1.204600,No Log
299,1.204600,1.258812
300,1.232800,No Log
399,1.232800,1.242484
400,1.414700,No Log
499,1.414700,33.628403
500,1.152400,No Log


[Step 100] Train Loss: 1.7673, Val Loss: 1.3354, Grad: 14.7699
[Step 200] Train Loss: 1.2046, Val Loss: 1.2758, Grad: 6.2409
[Step 300] Train Loss: 1.2328, Val Loss: 1.2588, Grad: 4.9408
[Step 400] Train Loss: 1.4147, Val Loss: 1.2425, Grad: 4.6263

 >>> [Pruning Event (No TA)] Step 500
     Stats - Loss: 1.1090, Grad: 2.9056 => Rate: 0.262
     Current Sparsity: 25.14% (40957402/162935040)

[Step 500] Train Loss: 1.1524, Val Loss: 33.6284, Grad: 3.1628
[Step 600] Train Loss: 3.1084, Val Loss: 1.9003, Grad: 20.1219
[Step 700] Train Loss: 1.8688, Val Loss: 1.8352, Grad: 3.1582
[Step 800] Train Loss: 2.2101, Val Loss: 1.7994, Grad: 3.3578
[Step 900] Train Loss: 1.8229, Val Loss: 1.7825, Grad: 2.6058

 >>> [Pruning Event (No TA)] Step 1000
     Stats - Loss: 2.0124, Grad: 2.6645 => Rate: 0.132
     Current Sparsity: 31.34% (51071378/162935040)

[Step 1000] Train Loss: 1.9275, Val Loss: 1.7711, Grad: 2.5612
[Step 1100] Train Loss: 2.1189, Val Loss: 1.7601, Grad: 2.5869
[Step 1200] Train Lo

✅ Final Test Loss: 1.4712
Finalizing model pruning...
✅ Done.
